# Conversation Threading: Practice Exercise

Build a customer support ticket system that handles **multi-user, multi-session conversations** using LangGraph checkpointers.

**What you'll implement:**
- A function to generate thread IDs that support multiple users with multiple sessions each
- A function to create the proper config dictionary for invoking the graph
- Test code demonstrating user isolation and separate session management

**Estimated time:** 10-15 minutes

## Setup

Run the cell below to import all required libraries and configure the environment.

In [ ]:
# Setup - run this cell first

import os
import sqlite3
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

print("Setup complete!")

## Context

You are building a **customer support ticket system** for an e-commerce company called ShopEasy. The system must handle:

1. **Multiple users**: Different customers should have completely isolated conversations (Customer A's issues should never appear in Customer B's threads)

2. **Multiple sessions per user**: Each customer can open multiple support tickets (sessions). For example:
   - Alice opens Ticket #1 about a shipping issue
   - Alice opens Ticket #2 about a refund request
   - These should be separate conversation threads, even though they're from the same customer

**Your challenge:** Design a thread ID scheme that enables:
- **User isolation**: Different customers never share threads
- **Session separation**: Same customer can have multiple independent conversation threads

**Input:**
- `user_id`: A unique identifier for each customer (e.g., "alice", "bob")
- `session_id`: An identifier for the specific support ticket/session (e.g., "ticket_001", "ticket_002")

**Expected behavior:**
- Same user, same session -> continues that specific conversation
- Same user, different session -> completely separate conversation (different ticket!)
- Different user -> completely separate conversation

## Provided Code

The following components are already implemented for you:
- State class with messages
- Support agent node with a customer service persona
- Compiled graph with SQLite persistence

In [ ]:
# State definition (provided)
class SupportTicketState(TypedDict):
    """State for the support ticket system."""
    messages: Annotated[list[BaseMessage], add_messages]

print("State defined!")

In [ ]:
# Support agent node (provided)
def support_agent_node(state: SupportTicketState) -> SupportTicketState:
    """Process customer messages and generate support responses."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    
    # Add system message for customer support persona
    system_msg = SystemMessage(content="""You are a helpful customer support agent for ShopEasy, 
an e-commerce company. Be friendly, professional, and remember details the customer 
has shared with you in this conversation (like their name, order numbers, or issues).""")
    
    messages_with_system = [system_msg] + state["messages"]
    response = llm.invoke(messages_with_system)
    
    return {"messages": [response]}

print("Support agent node defined!")

In [ ]:
# Build and compile graph with SQLite persistence (provided)
workflow = StateGraph(SupportTicketState)
workflow.add_node("support_agent", support_agent_node)
workflow.add_edge(START, "support_agent")
workflow.add_edge("support_agent", END)

# Create SQLite checkpointer
conn = sqlite3.connect("support_tickets.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

# Compile with checkpointer
support_app = workflow.compile(checkpointer=checkpointer)

print("Support ticket system compiled with SQLite persistence!")

## Part 1: Create the Thread ID Generator

Implement a function that generates unique thread IDs for multi-user multi-session conversations.

**Key insight:** For a support ticket system where:
- Different users must be isolated
- The same user can have multiple separate tickets/sessions

Think about what combination of information uniquely identifies each conversation thread.

In [ ]:
def generate_thread_id(user_id: str, session_id: str) -> str:
    """
    Generate a unique thread ID for the customer support ticket system.
    
    The thread ID must enable:
    - User isolation: Different users never share the same thread
    - Session separation: Same user with different sessions get different threads
    
    Args:
        user_id: Unique identifier for the customer (e.g., "alice", "bob")
        session_id: Identifier for the specific ticket/session (e.g., "ticket_001")
    
    Returns:
        A string thread_id that uniquely identifies this user's specific session.
        
    Requirements:
        - Different users must ALWAYS get different thread_ids
        - Same user with different session_ids must get different thread_ids
        - Same user with same session_id must get the same thread_id
    
    Example expected behavior:
        generate_thread_id("alice", "ticket_001") -> different from
        generate_thread_id("alice", "ticket_002") 
        (same user, different sessions = different threads)
        
        generate_thread_id("alice", "ticket_001") -> different from
        generate_thread_id("bob", "ticket_001")
        (different users = different threads)
        
        generate_thread_id("alice", "ticket_001") -> same as
        generate_thread_id("alice", "ticket_001")
        (same user, same session = same thread for continuity)
    """
    # TODO: Return a thread_id that uniquely identifies each user+session combination
    pass


# Test your implementation
if generate_thread_id("alice", "ticket_001") is not None:
    # Test 1: Same user, different sessions should get DIFFERENT thread_ids
    thread_a = generate_thread_id("alice", "ticket_001")
    thread_b = generate_thread_id("alice", "ticket_002")
    print(f"Alice, Ticket 001: {thread_a}")
    print(f"Alice, Ticket 002: {thread_b}")
    print(f"Different threads (session separation works): {thread_a != thread_b}")
    
    print()
    
    # Test 2: Different users should get different thread_ids
    thread_c = generate_thread_id("alice", "ticket_001")
    thread_d = generate_thread_id("bob", "ticket_001")
    print(f"Alice, Ticket 001: {thread_c}")
    print(f"Bob, Ticket 001: {thread_d}")
    print(f"Different threads (user isolation works): {thread_c != thread_d}")
    
    print()
    
    # Test 3: Same user, same session should get the SAME thread_id
    thread_e = generate_thread_id("alice", "ticket_001")
    thread_f = generate_thread_id("alice", "ticket_001")
    print(f"Alice, Ticket 001 (call 1): {thread_e}")
    print(f"Alice, Ticket 001 (call 2): {thread_f}")
    print(f"Same thread (continuity works): {thread_e == thread_f}")
else:
    print("Implement generate_thread_id() first, then run this cell again")

## Part 2: Create the Config Builder

When invoking a LangGraph with a checkpointer, you need to pass a config dictionary that specifies the thread_id. Implement a function that creates this config.

**The config structure for LangGraph:**
```python
config = {"configurable": {"thread_id": "your_thread_id_here"}}
```

This config is passed to `app.invoke()` to tell the checkpointer which conversation thread to load/save.

In [ ]:
def create_thread_config(thread_id: str) -> dict:
    """
    Create a config dictionary for invoking a LangGraph with a checkpointer.
    
    When using a checkpointer, the graph needs to know which thread to use.
    This is passed via a config dictionary with a specific structure.
    
    Args:
        thread_id: The unique identifier for the conversation thread
    
    Returns:
        A config dictionary with the structure:
        {"configurable": {"thread_id": <thread_id>}}
        
    Example:
        config = create_thread_config("alice_ticket_001")
        result = app.invoke({"messages": [...]}, config=config)
    """
    # TODO: Return a config dictionary with the thread_id in the correct structure
    pass


# Test your implementation
test_config = create_thread_config("test_thread_123")
if test_config is not None:
    print(f"Generated config: {test_config}")
    
    # Verify structure
    has_configurable = "configurable" in test_config
    has_thread_id = has_configurable and "thread_id" in test_config["configurable"]
    correct_value = has_thread_id and test_config["configurable"]["thread_id"] == "test_thread_123"
    
    print(f"Has 'configurable' key: {has_configurable}")
    print(f"Has 'thread_id' inside configurable: {has_thread_id}")
    print(f"Correct thread_id value: {correct_value}")
    print(f"Config structure is valid: {has_configurable and has_thread_id and correct_value}")
else:
    print("Implement create_thread_config() first, then run this cell again")

## Part 3: Demonstrate Multi-User Multi-Session Threading

Use your `generate_thread_id` and `create_thread_config` functions to demonstrate:
1. **User isolation**: Two customers (Alice and Bob) having separate conversations
2. **Multi-session per user**: Alice having two separate support tickets that don't mix

You'll need to:
1. Generate a thread_id using `generate_thread_id(user_id, session_id)`
2. Create a config using `create_thread_config(thread_id)`
3. Invoke the graph with `support_app.invoke({"messages": [HumanMessage(content=...)]}, config=config)`

In [ ]:
# TODO: Demonstrate multi-user multi-session conversation threading
#
# Your demonstration should show:
#
# 1. USER ISOLATION (two customers have separate conversations)
#    - Alice opens ticket_001 about a shipping problem with order #12345
#    - Bob opens ticket_001 about a refund for order #67890
#    - Ask each thread "What order number did I mention?"
#    - Verify Alice's thread says #12345, Bob's thread says #67890
#
# 2. MULTI-SESSION PER USER (same user, multiple separate tickets)
#    - Alice opens ticket_002 about a DIFFERENT issue - a damaged product from order #99999
#    - Ask Alice's ticket_002 "What order number did I mention?"
#    - Verify it says #99999 (NOT #12345 from her other ticket)
#    - Ask Alice's ticket_001 "What order number did I mention?"
#    - Verify it still says #12345 (tickets are separate!)
#
# For each message, you need to:
#   1. Generate thread_id: thread_id = generate_thread_id(user_id, session_id)
#   2. Create config: config = create_thread_config(thread_id)
#   3. Invoke graph: result = support_app.invoke({"messages": [HumanMessage(content="...")]}, config=config)
#   4. Get response: response = result["messages"][-1].content
# 
# Example:
#   thread_id = generate_thread_id("alice", "ticket_001")
#   config = create_thread_config(thread_id)
#   result = support_app.invoke({"messages": [HumanMessage(content="Hi!")]}, config=config)
#   print(result["messages"][-1].content)

print("=" * 70)
print("PART 3A: USER ISOLATION - Two customers, separate conversations")
print("=" * 70)

# TODO: Implement user isolation demonstration
# - Alice's ticket_001 conversation (shipping issue, order #12345)
# - Bob's ticket_001 conversation (refund request, order #67890)
# - Verify each thread remembers only its own order number


print("\n" + "=" * 70)
print("PART 3B: MULTI-SESSION - Alice has two separate support tickets")
print("=" * 70)

# TODO: Implement multi-session demonstration
# - Alice opens ticket_002 about damaged product, order #99999
# - Verify ticket_002 only knows about #99999
# - Verify ticket_001 still only knows about #12345

## Cleanup

Close the database connection when done.

In [ ]:
# Cleanup
conn.close()
print("Database connection closed.")

## Summary

In this exercise, you learned to design thread IDs and configs for multi-user multi-session scenarios:

**Key Insight 1 - Thread ID Design:** The thread_id determines what conversation history is loaded. For a support ticket system:

- **User isolation**: Different users must get different thread_ids
- **Session separation**: Same user with different sessions must get different thread_ids
- **Continuity**: Same user + same session must get the same thread_id

**Pattern:** Combine both `user_id` AND `session_id` to create unique thread identifiers:
```python
thread_id = f"{user_id}_{session_id}"
```

**Key Insight 2 - Config Structure:** LangGraph requires a specific config structure to pass the thread_id:
```python
config = {"configurable": {"thread_id": thread_id}}
result = app.invoke({"messages": [...]}, config=config)
```

The `configurable` key is required - LangGraph looks for `thread_id` inside this nested dictionary.

**Real-world applications:**
- Customer support ticket systems (each ticket is a separate thread)
- Project management tools (each project discussion is separate)
- Multi-tenant SaaS with per-user, per-workspace conversations
- Chat applications with multiple chat rooms per user